# Experiment 8: LoRA transfer and retention

How do low-rank query/value updates affect predictive quality on excluded credit tables, and what happens to non-credit retention? Report every planned learning rate against its own original checkpoint. The five-fold benchmark uses matched inference settings within this experiment, including FP32 without FA3 for TabICL. The earlier full/frozen grid stays separate unless its protocol and artifact provenance match.

## 0. Evidence and scope

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, lora_study as ls, publication as pub
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
sink = FigureSaver('experiment8/02_results')
report = cp.NotebookReport('Experiment 8: LoRA transfer and retention', sink=sink)
print(source_description())
report.add('0. Evidence and scope', source_description() + '\n' + 'Original versus LoRA comparisons describe the fixed experiment-8 recipe; no best-rate selection is performed on the reported held-out scores.')
runs = [cp.load_campaign(8, track) for track in ('pd', 'lgd')]
from src.visualize import comparisons as cmp, dashboards as db


DATA: no external source selected.
Project: no external source selected.
Missing project records are unavailable evidence, not zero effects or failed training.


## 1. Checkpoint and paired evaluation coverage

In [2]:
for run in runs:
    report.table(run.track.upper()+' protocol', ls.protocol_table(run))
    report.table(run.track.upper()+' evidence', pub.availability(run))
    report.table(run.track.upper()+' planned and recorded trials', ls.coverage_table(run))
    report.table(run.track.upper()+' paired benchmark coverage', cmp.benchmark_coverage(run))
report.add('1. Checkpoint and paired evaluation coverage', 'Every trained/original comparison requires all five finite outer folds. Non-credit tables also require all four training partitions. A pending roster or absent figure is not a zero effect and does not indicate whether a remote job is running.')

,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,PD
2,Adaptation,Low-rank updates to all attention query/value ...
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Factor weight decay,0.0
7,Training precision,AMP (BF16 on B200); LoRA factors and BA produc...
8,Learning rates,"1e-05, 0.0001, 0.001"
9,L2-SP,0


,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,0,DATA manifests
2,Epoch rows,0,Project training diagnostics
3,Milestone rows,0,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,base,learning_rate,planned,completed,recorded_outcomes,evidence
0,TabICLv2,0.00001,4,0,PENDING: 4,No recorded training outcome
1,TabICLv2,0.00010,4,0,PENDING: 4,No recorded training outcome
2,TabICLv2,0.00100,4,0,PENDING: 4,No recorded training outcome
3,TabPFN v3,0.00001,4,0,PENDING: 4,No recorded training outcome
4,TabPFN v3,0.00010,4,0,PENDING: 4,No recorded training outcome
5,TabPFN v3,0.00100,4,0,PENDING: 4,No recorded training outcome


,domain,base,planned_checkpoints,completed_checkpoints,paired_five_fold_comparisons,paired_datasets
0,credit,TabICLv2,12,0,0,0
1,credit,TabPFN v3,12,0,0,0
2,ood,TabICLv2,12,0,0,0
3,ood,TabPFN v3,12,0,0,0


,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,LGD
2,Adaptation,Low-rank updates to all attention query/value ...
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Factor weight decay,0.0
7,Training precision,AMP (BF16 on B200); LoRA factors and BA produc...
8,Learning rates,"1e-05, 0.0001, 0.001"
9,L2-SP,0


,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,0,DATA manifests
2,Epoch rows,0,Project training diagnostics
3,Milestone rows,0,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,base,learning_rate,planned,completed,recorded_outcomes,evidence
0,TabICLv2,0.00001,4,0,PENDING: 4,No recorded training outcome
1,TabICLv2,0.00010,4,0,PENDING: 4,No recorded training outcome
2,TabICLv2,0.00100,4,0,PENDING: 4,No recorded training outcome
3,TabPFN v3,0.00001,4,0,PENDING: 4,No recorded training outcome
4,TabPFN v3,0.00010,4,0,PENDING: 4,No recorded training outcome
5,TabPFN v3,0.00100,4,0,PENDING: 4,No recorded training outcome


,domain,base,planned_checkpoints,completed_checkpoints,paired_five_fold_comparisons,paired_datasets
0,credit,TabICLv2,12,0,0,0
1,credit,TabPFN v3,12,0,0,0
2,ood,TabICLv2,12,0,0,0
3,ood,TabPFN v3,12,0,0,0


## 2. Original and LoRA predictive quality

AUC for PD and R² for LGD are shown as equal-table means against learning rate. Each panel uses the same available table cohort at all three rates; credit transfer and non-credit retention remain separate.

In [3]:
for run in runs:
    cp.show(sink, ls.plot_primary(run))
    for domain in ('credit', 'ood'):
        report.table(run.track.upper()+' '+domain+' paired primary scores', ls.benchmark_tables(run, domain=domain))
report.add('2. Original and LoRA predictive quality', 'The original checkpoint supplies a matched reference at every LoRA learning rate. The source tables retain both absolute scores and paired effects. The plots connect rate settings, not training time. Table means are descriptive; the best observed rate is not a validated selected recipe.')

No measurements available for this section yet.


""


""


No measurements available for this section yet.


""


""


## 3. Probability quality and regression errors

Brier score and log loss complement PD ranking performance. RMSE and MAE complement LGD R². Positive plotted changes consistently indicate improvement.

In [4]:
for run in runs:
    cp.show(sink, ls.plot_secondary(run))
    for metric in (('brier_score', 'log_loss', 'ece', 'f1') if run.track == 'pd' else ('rmse', 'mae')):
        for domain in ('credit', 'ood'):
            report.table(run.track.upper()+' '+domain+' '+metric, ls.benchmark_tables(run, metric, domain))
report.add('3. Probability quality and regression errors', 'Each secondary figure requires the same complete table cohort for every rate and both displayed metrics. Bands show table interquartile ranges, not confidence intervals. RMSE changes are fractional reductions; Brier, log-loss and MAE differences remain on their native scales. Missing or nonfinite values are never filled with zero.')

No measurements available for this section yet.


""


""


""


""


...

No measurements available for this section yet.


""


""


""


""


## 4. Cost and comparison with earlier adaptation recipes

In [5]:
for run in runs:
    report.table(run.track.upper()+' training resources', ls.costs(run))
    reference_cfg = cp.load_train_config(config_path=str(cp.config_path(1, run.track)))
    reference = cp.Campaign(reference_cfg, cp.planned_trials(reference_cfg), run.trajectories.iloc[:0], run.histories.iloc[:0], run.evaluation.iloc[:0])
    report.table(run.track.upper()+' experiment-1 protocol comparison', ls.reference_protocol_audit(run, reference))
report.add('4. Cost and comparison with earlier adaptation recipes', 'Only learning rate 1e-5 overlaps the earlier L2-SP-zero full/frozen grid. In particular, TabICL benchmark precision differs from experiment 1. Earlier scores are therefore not silently pooled into a parameter-count ablation. A direct comparison requires matching base/data hashes, row folds, inference settings and remaining training protocol. Rank and alpha are fixed, so these results do not estimate the best achievable LoRA performance. Overlapping training corpora, related sources and the small non-credit panel limit independence and generality.')

,trial_name,base,learning_rate,partition,seed,status
0,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00001,0,42,PENDING
1,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00010,0,42,PENDING
2,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00100,0,42,PENDING
3,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00001,0,42,PENDING
4,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00010,0,42,PENDING
5,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00100,0,42,PENDING
6,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00001,1,42,PENDING
7,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00010,1,42,PENDING
8,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00100,1,42,PENDING
9,cpt_lora_v1_s01_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00001,1,42,PENDING


,check,experiment1,experiment8,matches
0,seed,42,42,True
1,train.target_total_steps,10000,10000,True
2,train.max_rows_per_step,None,None,True
3,train.context_sampling,balanced,balanced,True
4,train.amp,True,True,True
5,optimizer.weight_decay,0.0,0.0,True
6,scheduler,"{'warmup_fraction': 0.1, 'min_lr_fraction': 0.05}","{'warmup_fraction': 0.1, 'min_lr_fraction': 0.05}",True
7,corpus.n_folds,4,4,True
8,corpus.split_seed,1729,1729,True
9,evaluation.tabicl_inference_options,None,"{'use_amp': False, 'use_fa3': False}",False


,trial_name,base,learning_rate,partition,seed,status
0,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00001,0,42,PENDING
1,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00010,0,42,PENDING
2,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00100,0,42,PENDING
3,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00001,0,42,PENDING
4,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00010,0,42,PENDING
5,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00100,0,42,PENDING
6,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00001,1,42,PENDING
7,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00010,1,42,PENDING
8,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00100,1,42,PENDING
9,cpt_lora_v1_s01_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00001,1,42,PENDING


,check,experiment1,experiment8,matches
0,seed,42,42,True
1,train.target_total_steps,10000,10000,True
2,train.max_rows_per_step,None,None,True
3,train.context_sampling,stratified,stratified,True
4,train.amp,True,True,True
5,optimizer.weight_decay,0.0,0.0,True
6,scheduler,"{'warmup_fraction': 0.1, 'min_lr_fraction': 0.05}","{'warmup_fraction': 0.1, 'min_lr_fraction': 0.05}",True
7,corpus.n_folds,4,4,True
8,corpus.split_seed,1729,1729,True
9,evaluation.tabicl_inference_options,None,"{'use_amp': False, 'use_fa3': False}",False


## Complete text summary

All displayed values, tables and captions in section order.

In [6]:
print(report.summary(sink))

Experiment 8: LoRA transfer and retention

0. Evidence and scope
DATA: no external source selected.
Project: no external source selected.
Missing project records are unavailable evidence, not zero effects or failed training.
Original versus LoRA comparisons describe the fixed experiment-8 recipe; no best-rate selection is performed on the reported held-out scores.

1. Checkpoint and paired evaluation coverage
Every trained/original comparison requires all five finite outer folds. Non-credit tables also require all four training partitions. A pending roster or absent figure is not a zero effect and does not indicate whether a remote job is running.

Table: PD protocol
CSV (first column: row index)
row,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,PD
2,Adaptation,Low-rank updates to all attention query/value projections; base weights fixed
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Factor weight decay,0.0
7,Training precision,AMP (BF16 on B200); LoRA factors and BA product FP32
8,Lea